In [ ]:
!pip install findspark

# Agregaciones en Apache Spark

La realización de análisis profundos sobre Big Data generalmente implica algún tipo de **agregación** para resumir los datos, con el fin de extraer patrones, obtener conocimientos de negocio o generar informes ejecutivos condensados.

Las agregaciones normalmente requieren alguna forma de **agrupación**, ya sea en todo el conjunto de datos o basándose en una o más columnas específicas. Una vez agrupados los datos, se aplican funciones matemáticas de agregación como sumar (`sum`), contar (`count`) o promediar (`avg`) a cada grupo. Spark proporciona una amplia variedad de estas funciones integradas y altamente optimizadas, así como la capacidad de agregar valores en una colección para su posterior análisis detallado.

En Spark, **todas las agregaciones se realizan a través de funciones**. Estas funciones de agregación operan sobre un conjunto determinado de filas, independientemente de si ese conjunto abarca la totalidad de los registros del DataFrame o únicamente un subgrupo específico de ellos.

---

## Niveles de Agrupación en Spark

La agrupación de filas se puede realizar a diferentes niveles de granularidad. Spark admite los tres enfoques principales siguientes:

### 1. Tratar un DataFrame completo como un único grupo
Consiste en aplicar una función de agregación global sobre la totalidad de los datos. No se especifica ninguna columna de separación y el resultado devuelve una única fila resumida.
* *Ejemplo:* Contar el total de registros del DataFrame o calcular el promedio de ingresos globales.

### 2. Agrupación por columnas (`groupBy`)
Permite dividir un DataFrame en múltiples subgrupos utilizando los valores de una o más columnas. Posteriormente, se calcula una o varias agregaciones de forma independiente para cada uno de esos grupos.
* *Ejemplo:* Agrupar un DataFrame de ventas por las columnas `País` y `Categoría` para obtener la suma total vendida en cada combinación.

### 3. Funciones de Ventana (`Window Functions`)
Consiste en dividir un DataFrame en varias "ventanas" o secciones de datos lógicas (basadas en particiones y ordenamientos), permitiendo realizar cálculos avanzados sobre un conjunto de filas que están directamente relacionadas con la fila actual. A diferencia del `groupBy` tradicional, las funciones de ventana no combinan ni reducen las filas, sino que mantienen la identidad de cada registro individual.
* *Ejemplo:* Calcular una **media móvil**, una **suma acumulativa** a lo largo del tiempo o una **clasificación/ranking** de los mejores empleados por departamento.

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Spark Agregaciones ").getOrCreate()

In [ ]:
df = spark.read.parquet("/content/data_vuelo")

In [ ]:
df.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

In [ ]:
df.show(20, truncate=False)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-